# Video Generasiyası — Hugging Face (Diffusers)

Bu notebook mətndən qısa video (text-to-video) generasiyasını göstərir.

## Text-to-video necə işləyir?

Əsas fikir şəkil generasiyası ilə eynidir (diffusion), sadəcə tək şəkil yox, zamanla ardıcıl və bir-birinə bağlı bir neçə kadr (frame) generasiya olunur ki, hərəkət hissi yaransın. Buna görə video generasiyası şəkil generasiyasından qat-qat ağırdır — hər kadr üçün diffusion prosesi işləyir.

> **Colab GPU:** İşə başlamazdan əvvəl: `Runtime → Change runtime type → GPU (T4)` seç. GPU olmadan bu modellər ya işləməyəcək, ya da çox yavaş olacaq.

Video generasiyası şəkildən qat-qat çox yaddaş istəyir, ona görə `enable_model_cpu_offload()` mütləq istifadə olunur.

In [ ]:
!pip install -q diffusers transformers accelerate imageio imageio-ffmpeg

In [ ]:
import torch
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
from diffusers.utils import export_to_video

### Modeli yükləmək
`damo-vilab/text-to-video-ms-1.7b` — free T4-də (16GB) işləyən nisbətən yüngül video modeli.

In [ ]:
pipe = DiffusionPipeline.from_pretrained(
    "damo-vilab/text-to-video-ms-1.7b",
    torch_dtype=torch.float16,
    variant="fp16"
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()  # T4-də yaddaşa sığmaq üçün vacib

### Video generasiyası

In [ ]:
prompt = "a cat surfing on a wave, cinematic"
negative_prompt = "low quality, blurry"

video_frames = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=25,
    num_frames=24
).frames

video_path = export_to_video(video_frames)
print("Video saxlanıldı:", video_path)

### Colab-da videoya baxmaq

In [ ]:
from IPython.display import Video
Video(video_path, embed=True)

### Parametrlər nə deməkdir?

- `num_frames` — video neçə kadrdan ibarət olacaq (model ~8 kadr/saniyə sürətlə render edir, 24 kadr ≈ 3 saniyə)
- `num_inference_steps` — kadr başına diffusion addımı, 20–30 kifayətdir
- Daha uzun/keyfiyyətli video istəsən `num_frames`-i artır, amma vaxt və yaddaş xətti şəkildə artır

## Növbəti addımlar

- Daha keyfiyyətli nəticə üçün `Wan2.1` və ya `AnyFlow` kimi yeni modellərə bax — amma bunlar A100 səviyyəli GPU istəyir (Colab Pro+ lazımdır)
- `image-to-video` pipeline-ları ilə statik şəkli canlandırmaq olar
- Render vaxtını azaltmaq üçün `num_inference_steps`-i 15-ə endirib sınamaq olar (keyfiyyətdən azca güzəştə dəyər)